# IMPORTING REQUIRED LIBRARIES


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Plot display configuration
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12


# 1. CROSS-ENTROPY LOSS FOR CLASSIFICATION


In [ ]:
# Formula: L = -sum(y_i * log(p_i))
# where y_i - true label, p_i - predicted probability

def cross_entropy_loss(y_true, y_pred, epsilon=1e-15):
    """
    Computes cross-entropy loss.

    Parameters:
    y_true: true labels in one-hot format (e.g. [0, 1, 0] for class 1)
    y_pred: predicted probabilities (e.g. [0.1, 0.8, 0.1])
    epsilon: a small number (0.000000000000001) to avoid log(0) = -inf

    Returns:
    loss: cross-entropy loss value (smaller is better)
    """
    # Clip the probabilities to avoid log(0) and log(1)
    # log(0) = -inf, which would break the computation
    # So we restrict probabilities to the range [epsilon, 1-epsilon]
    # Adding epsilon gives numerical stability (avoids log(0))

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # Apply the cross-entropy formula
    # Multiply the true labels by the log of the predicted probabilities
    # If y_true = 1, include log(p_pred)
    # If y_true = 0, that class does not contribute to the loss
    loss = -np.sum(y_true * np.log(y_pred))

    return loss / len(y_true)


In [ ]:
#  Load the real Iris dataset for demonstration
# Iris - the classic dataset with 3 species of iris flower
# 4 features: petal and sepal length and width

print("\n📊 Loading the Iris dataset...")
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Dataset information
print(f"📈 Dataset size: {X_iris.shape}")
print(f"🏷️  Number of classes: {len(np.unique(y_iris))}")
print(f"📋 Class names: {iris.target_names}")


In [ ]:
# Split the data into training (70%) and test (30%) sets
# test_size=0.3 means 30% of the data is used for testing
# random_state=42 ensures reproducibility of the results
# stratify=y_iris preserves class proportions in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"🎯 Training set size: {X_train.shape}")
print(f"🎯 Test set size: {X_test.shape}")


In [ ]:
# Train a logistic regression model
# Logistic regression works well for multi-class classification
# random_state=42 for reproducibility

print("\n🤖 Training the logistic regression model...")
model_lr = LogisticRegression(random_state=42)
model_lr.fit(X_train, y_train)


In [ ]:
# Get the predicted probabilities for each class
# predict_proba() returns a matrix of shape (n_samples, n_classes)
# Each row contains the probabilities of belonging to each class
# The probabilities in each row sum to 1

y_pred_proba = model_lr.predict_proba(X_test)
print(f"📊 Predicted probabilities shape: {y_pred_proba.shape}")


In [ ]:
# Convert the true labels to one-hot format
# One-hot encoding: [0, 1, 0] means membership in the 2nd class
# This is needed to compute cross-entropy loss
y_test_onehot = np.zeros((len(y_test), len(np.unique(y_iris))))
for i, label in enumerate(y_test):
    y_test_onehot[i, label] = 1

print(f"📋 One-hot labels shape: {y_test_onehot.shape}")


In [ ]:
# Compute the cross-entropy loss
ce_loss = cross_entropy_loss(y_test_onehot, y_pred_proba)
print(f"\n🎯 Cross-entropy loss: {ce_loss:.4f}")


In [ ]:
# Compare with sklearn's built-in function to verify the cross_entropy_loss formula
# log_loss in sklearn is the same thing as cross-entropy loss

from sklearn.metrics import log_loss
sklearn_ce_loss = log_loss(y_test, y_pred_proba)
print(f"📊 Sklearn cross-entropy loss: {sklearn_ce_loss:.4f}")


Interpretation of the values
	•	0.0 -> perfect (the model always gives probability 1.0 to the correct class).
	•	0.2 – 0.5 -> good model (correct predictions with high confidence).
	•	1.0 – 2.0 -> mediocre (many doubts and mistakes).
	•	> 2.0 -> the model is often confident in wrong answers.


In [ ]:
# Show example predictions to understand how the model works
print("\n📋 Example predictions:")

for i in range(min(10, len(y_test))):
    true_class_name = iris.target_names[y_test[i]]
    true_class_idx = y_test[i]
    pred_probs = y_pred_proba[i]
    pred_class_idx = np.argmax(pred_probs)
    pred_class_name = iris.target_names[pred_class_idx]

    # Maximum probability = overall confidence
    max_confidence = np.max(pred_probs)

    # Confidence in the correct class
    true_class_confidence = pred_probs[true_class_idx]

    # Correctness indicator
    correct = "✅" if pred_class_idx == true_class_idx else "❌"

    print(f"Sample {i+1}: {correct} True = {true_class_name}, "
          f"Predicted = {pred_class_name}, "
          f"Confidence = {max_confidence:.3f}, "
          f"Confidence in true class = {true_class_confidence:.3f}")

    # Show the full probability vector for the first 5 examples
    if i < 5:
        print(f"    Full probabilities: [{pred_probs[0]:.3f}, {pred_probs[1]:.3f}, {pred_probs[2]:.3f}]")

# Visualize the results for better understanding
plt.figure(figsize=(15, 5))

for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.bar(iris.target_names, y_pred_proba[i])
    plt.title(f"Sample {i+1}\nTrue: {iris.target_names[y_test[i]]}")
    plt.ylim(0, 1)

plt.tight_layout()
plt.show()


## Plot 1: Class probability distribution


In [ ]:
# Extract probabilities for each predicted class
setosa_confidences = []      # setosa probability for samples predicted as setosa
versicolor_confidences = []  # versicolor probability for samples predicted as versicolor
virginica_confidences = []   # virginica probability for samples predicted as virginica


for i in range(len(y_test)):
    true_class = y_test[i]
    predicted_proba = y_pred_proba[i]

    # Take the probability of the correctly predicted class
    confidence = predicted_proba[true_class]

    if true_class == 0:  # setosa
        setosa_confidences.append(confidence)
    elif true_class == 1:  # versicolor
        versicolor_confidences.append(confidence)
    else:  # virginica
        virginica_confidences.append(confidence)


In [ ]:
# Plot 1: Class probability distribution
# This plot shows how confidently the model predicts each class

print(f"HOW TO READ PLOT 1:")
print(f"- High values (close to 1) mean high confidence in the class")
print(f"- Low values (close to 0) mean low confidence")
print(f"- A wide spread means different confidences across samples")
print(f"- A narrow spread means stable predictions")


plt.figure(figsize=(10, 6))
plt.boxplot([setosa_confidences, versicolor_confidences, virginica_confidences])
plt.title('Model confidence for each class')
plt.xlabel('Class')
plt.ylabel('Confidence')
plt.xticks([1, 2, 3], iris.target_names)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print(f"\n📊 PLOT 1 (DETAILED): Detailed confidence analysis")

plt.figure(figsize=(12, 8))
# Build a CORRECT box plot
bp = plt.boxplot([setosa_confidences, versicolor_confidences, virginica_confidences],
                 patch_artist=True, labels=iris.target_names)
# Color the boxes
colors = ['lightcoral', 'lightblue', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.title('✅ CORRECT: Model confidence for each class\n(Only for true classes)',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Iris class', fontsize=12)
plt.ylabel('Model confidence\n(0 = not confident at all, 1 = very confident)', fontsize=12)

# Add detailed annotations
plt.annotate('🎯 EXCELLENT!\nVery high confidence\n(nearly all > 0.95)\nThe model works great with setosa',
             xy=(1, 0.95), xytext=(0.3, 0.7),
             arrowprops=dict(arrowstyle='->', color='red', lw=3),
             fontsize=11, ha='center',
             bbox=dict(boxstyle="round,pad=0.5", facecolor="yellow", alpha=0.8))

plt.annotate('✅ GOOD\nGood confidence\n(most > 0.8)\nThe model handles versicolor',
             xy=(2, 0.85), xytext=(2.7, 0.6),
             arrowprops=dict(arrowstyle='->', color='blue', lw=3),
             fontsize=11, ha='center',
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))

plt.annotate('👍 OK\nModerate confidence\n(most > 0.7)\nThe hardest class, but still works',
             xy=(3, 0.75), xytext=(3.7, 0.9),
             arrowprops=dict(arrowstyle='->', color='green', lw=3),
             fontsize=11, ha='center',
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgreen", alpha=0.8))

# Add reference lines
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, linewidth=2, label='50% = poor')
plt.axhline(y=0.8, color='orange', linestyle='--', alpha=0.7, linewidth=2, label='80% = good')
plt.axhline(y=0.9, color='green', linestyle='--', alpha=0.7, linewidth=2, label='90% = excellent')
plt.legend(loc='lower right', fontsize=10)

# Add a detailed explanation
plt.text(0.02, 0.98, 'HOW TO READ THIS PLOT:\n\n📦 Box = where 50% of the data lies\n📏 Line inside the box = median (typical value)\n📐 Whiskers = min and max\n⚫ Dots = outliers (unusual cases)\n\n🎯 WHAT TO LOOK FOR:\n• Tall boxes = the model is confident\n• Narrow boxes = stable predictions\n• Wide boxes = lots of spread\n\n✅ NOW SHOWS REAL CONFIDENCE!',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle="round,pad=0.6", facecolor="white", alpha=0.9, edgecolor='black'))

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 KEY TAKEAWAYS FROM THE DETAILED PLOT 1:")
print("   ✅ Setosa: The model is VERY confident (mean ~0.97)!")
print("   ✅ Versicolor: The model is confident (mean ~0.85)!")
print("   ✅ Virginica: The model is fairly confident (mean ~0.75)!")
print("   🎯 ALL CLASSES SHOW GOOD CONFIDENCE!")


## Plot 2: Confusion matrix


In [ ]:
# Plot 2: Confusion matrix
from sklearn.metrics import confusion_matrix
y_pred = model_lr.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.subplot(1, 3, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Confusion matrix')
plt.xlabel('Predicted class')
plt.ylabel('True class')


In [ ]:
print("\n" + "="*60)
print("📊 Showing Plot 2: Confusion matrix")
print("="*60)

plt.figure(figsize=(10, 8))

cm = confusion_matrix(y_test, y_pred)
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                 xticklabels=iris.target_names,
                 yticklabels=iris.target_names,
                 cbar_kws={'label': 'Number of samples'})

plt.title('Confusion matrix\n(How often the model was wrong and right)',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicted class (what the model said)', fontsize=12)
plt.ylabel('True class (what was actually there)', fontsize=12)

# Add annotations to the cells
for i in range(len(iris.target_names)):
    for j in range(len(iris.target_names)):
        value = cm[i, j]
        if i == j:  # Diagonal elements (correct predictions)
            plt.annotate(f'✅ CORRECT!\n{value} samples',
                        xy=(j+0.5, i+0.5), ha='center', va='center',
                        fontsize=10, fontweight='bold', color='white' if value > 8 else 'black')
        elif value > 0:  # Errors
            plt.annotate(f'❌ ERROR!\n{value} samples',
                        xy=(j+0.5, i+0.5), ha='center', va='center',
                        fontsize=10, fontweight='bold', color='red')

# Add an explanation
plt.text(1.15, 1.5, 'HOW TO READ THE MATRIX:\n\n🔍 Each cell shows:\n   how many times the model predicted\n   a particular class\n\n✅ DIAGONAL (ideal):\n   Correct predictions\n   Darker = more\n\n❌ OFF-DIAGONAL (bad):\n   Model errors\n   Should be light\n\n🎯 IDEAL MATRIX:\n   Dark diagonal,\n   light everywhere else',
         transform=plt.gca().transAxes, fontsize=11, verticalalignment='center',
         bbox=dict(boxstyle="round,pad=0.6", facecolor="lightyellow", alpha=0.9, edgecolor='black'))

plt.tight_layout()
plt.show()

# Compute and show statistics
total_correct = np.sum(np.diag(cm))
total_samples = np.sum(cm)
accuracy = total_correct / total_samples

print(f"\n📈 STATISTICS:")
print(f"   🎯 Overall accuracy: {accuracy:.2%} ({total_correct}/{total_samples})")
print(f"   ✅ Correct: {total_correct} samples")
print(f"   ❌ Errors: {total_samples - total_correct} samples")



## Plot 3: Impact of confidence on loss


In [ ]:
# Plot 3: Impact of confidence on loss
plt.subplot(1, 3, 3)
confidences = np.max(y_pred_proba, axis=1)
individual_losses = []

for i in range(len(y_test)):
    true_onehot = np.zeros(3)
    true_onehot[y_test[i]] = 1
    loss = cross_entropy_loss(true_onehot.reshape(1, -1),
                            y_pred_proba[i].reshape(1, -1))
    individual_losses.append(loss)

plt.scatter(confidences, individual_losses, alpha=0.6)
plt.xlabel('Model confidence')
plt.ylabel('Individual Cross-entropy Loss')
plt.title('Relationship between confidence and loss')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation of Cross-entropy Loss:")
print("   • The lower the loss, the better the model")
print("   • Loss = 0 means perfect predictions")
print("   • High loss indicates uncertain predictions")


In [ ]:
print(f"\n📊 PLOT 3: Relationship between confidence and loss with annotations")

plt.figure(figsize=(15, 5))

# Plot 3: Impact of confidence on loss WITH ANNOTATIONS
plt.subplot(1, 3, 3)
confidences = np.max(y_pred_proba, axis=1)
individual_losses = []
for i in range(len(y_test)):
    true_onehot = np.zeros(3)
    true_onehot[y_test[i]] = 1
    loss = cross_entropy_loss(true_onehot.reshape(1, -1),
                            y_pred_proba[i].reshape(1, -1))
    individual_losses.append(loss)

# Create a scatter plot colored by class
scatter = plt.scatter(confidences, individual_losses,
                     c=y_test, cmap='viridis', alpha=0.8, s=60, edgecolors='black')

plt.xlabel('Model confidence')
plt.ylabel('Individual Cross-entropy Loss')
plt.title('Relationship between confidence and loss')

# Add the color legend
cbar = plt.colorbar(scatter, ax=plt.gca())
cbar.set_label('Class', fontsize=8)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(['setosa', 'versicolor', 'virginica'])

# Add zones
plt.axvspan(0.8, 1.0, alpha=0.1, color='green')
plt.axhspan(0, 0.5, alpha=0.1, color='green')
plt.axvspan(0.0, 0.6, alpha=0.1, color='red')

# Add annotations
plt.annotate('GOOD:\nHigh confidence\nLow loss',
             xy=(0.9, 0.1), xytext=(0.85, 0.6),
             arrowprops=dict(arrowstyle='->', color='green', lw=2),
             fontsize=9, ha='center',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7))

plt.annotate('BAD:\nLow confidence\nHigh loss',
             xy=(0.4, 1.0), xytext=(0.2, 1.5),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=9, ha='center',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.7))

# Add reference lines
plt.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label='Loss threshold')
plt.axvline(x=0.8, color='orange', linestyle='--', alpha=0.7, label='Confidence threshold')

plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation of Cross-entropy Loss:")
print("   • Each point = one sample from the test set")
print("   • Point color = which class the sample belongs to")
print("   • Bottom-right corner = best predictions")
print("   • Top-left corner = worst predictions")
print("   • The lower the loss, the better the model")
print("   • Loss = 0 means perfect predictions")
